In [1]:
from pathlib import Path
import json
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

ROOT = Path.cwd().resolve()
if ROOT.name == "linear_regression":
    ROOT = ROOT.parent

TRAIN_FILE = ROOT / "data/processed/Hanoi_final.csv"
TEST_FILE = ROOT / "data/processed/Hanoi_test_final.csv"
OUT = ROOT / "linear_regression" / "ln_output"
OUT.mkdir(parents=True, exist_ok=True)

WINDOW = 24
TARGET = "temperature_2m"
SEED = 42
np.random.seed(SEED)

In [2]:
# Danh sách 32 features
weather_features = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "apparent_temperature", "precipitation", "rain", "pressure_msl",
    "surface_pressure", "cloud_cover", "wind_speed_10m", "wind_speed_100m",
    "wind_direction_10m_sin", "wind_direction_10m_cos",
    "wind_direction_100m_sin", "wind_direction_100m_cos", "wind_gusts_10m",
    "et0_fao_evapotranspiration", "vapour_pressure_deficit",
    "soil_temperature_0_to_7cm", "soil_temperature_7_to_28cm",
    "soil_temperature_28_to_100cm", "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm", "soil_moisture_100_to_255cm",
    "weather_code", "is_day",
]
features = weather_features + ["hour_sin", "hour_cos", "day_of_year_sin", "day_of_year_cos"]

In [3]:
# Đọc dữ liệu 
data = pd.read_csv(TRAIN_FILE , parse_dates = ['time']).sort_values('time').reset_index(drop = True)

values = data[features].to_numpy(dtype = np.float32)
temperature = data[TARGET].to_numpy(dtype = np.float32)

#Tạo Sliding Window
windows = sliding_window_view(values , WINDOW , axis = 0)[:-1]
windows = windows.transpose(0,2,1)[: , ::-1 , :]

X = windows.reshape(len(windows),-1).copy()
y = temperature[WINDOW:]

times = pd.DatetimeIndex(data['time'].iloc[WINDOW:])
baseline = temperature[WINDOW -1 : -1]


# Chia train / validation
train_count = (times < "2025-01-01").sum()
X_train, y_train = X[:train_count], y[:train_count]
X_valid, y_valid = X[train_count:], y[train_count:]
valid_baseline = baseline[train_count:]

print("Train:", X_train.shape, "| Validation:", X_valid.shape)

Train: (131448, 768) | Validation: (8760, 768)


In [4]:
def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    r2 = 1.0 - (np.sum((y_true - y_pred) ** 2) / (np.sum((y_true - np.mean(y_true)) ** 2) + 1e-12))
    return {"MAE": float(mae), "RMSE": float(rmse), "R2": float(r2)}

# Chuẩn hóa Z-score trực tiếp bằng biến NumPy (chỉ lấy mean, std từ X_train)
x_mean = np.mean(X_train, axis=0, dtype=np.float64)
x_std = np.std(X_train, axis=0, dtype=np.float64)
x_std[x_std < 1e-8] = 1.0  # Tránh chia cho 0

X_train_scaled = ((X_train - x_mean) / x_std).astype(np.float32)
X_valid_scaled = ((X_valid - x_mean) / x_std).astype(np.float32)

print("Persistence (validation):", evaluate(y_valid, valid_baseline))

Persistence (validation): {'MAE': 0.5569748978636581, 'RMSE': 0.8955298743336904, 'R2': 0.9737443307300588}


In [5]:
def train_linear_gd(X_tr, y_tr, X_val, y_val, l1=0.0, l2=0.0, lr=0.005, epochs=100, batch_size=2048, patience=10):
    N, D = X_tr.shape
    rng = np.random.default_rng(SEED)
    
    # Khởi tạo trọng số w = 0, hệ số chặn b = trung bình của y_tr
    w = np.zeros(D, dtype=np.float64)
    b = float(np.mean(y_tr))
    
    best_mae = float("inf")
    best_w, best_b = w.copy(), b
    best_epoch = 0
    wait = 0

    for epoch in range(1, epochs + 1):
        indices = rng.permutation(N)
        
        for start in range(0, N, batch_size):
            batch_idx = indices[start : start + batch_size]
            Xb = X_tr[batch_idx].astype(np.float64)
            yb = y_tr[batch_idx].astype(np.float64)
            m = len(batch_idx)

            # 1. Dự đoán & Tính sai số
            error = (Xb @ w + b) - yb

            # 2. Đạo hàm (MSE + L1 + L2)
            grad_w = (Xb.T @ error) / m + l1 * np.sign(w) + l2 * w
            grad_b = np.mean(error)

            # 3. Cập nhật w, b
            w -= lr * grad_w
            b -= lr * grad_b

        # Kiểm tra trên tập Validation cuối mỗi epoch
        val_preds = X_val @ w + b
        val_mae = np.mean(np.abs(y_val - val_preds))

        if val_mae < best_mae - 1e-5:
            best_mae = val_mae
            best_w, best_b = w.copy(), b
            best_epoch = epoch
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    return best_w.astype(np.float32), float(best_b), best_epoch

In [ ]:
lr_configs = [
    {"name": "OLS",            "l1": 0.0,  "l2": 0.0},
    {"name": "Ridge_L2_0.001", "l1": 0.0,  "l2": 1e-3},
    {"name": "Ridge_L2_0.01",  "l1": 0.0,  "l2": 1e-2},
    {"name": "Ridge_L2_0.1",   "l1": 0.0,  "l2": 1e-1},
    {"name": "Lasso_L1_0.0001","l1": 1e-4, "l2": 0.0},
    {"name": "Lasso_L1_0.001", "l1": 1e-3, "l2": 0.0},
    {"name": "Lasso_L1_0.005", "l1": 5e-3, "l2": 0.0},
    {"name": "Elastic_L1_L2",  "l1": 1e-3, "l2": 1e-2},
]

lr_rows = []
saved_weights = {}  # Lưu trực tiếp cặp (w, b) của từng cấu hình

for i, cfg in enumerate(lr_configs):
    start = perf_counter()
    w, b, best_ep = train_linear_gd(
        X_train_scaled, y_train, 
        X_valid_scaled, y_valid, 
        l1=cfg["l1"], l2=cfg["l2"], lr=0.005, epochs=100
    )
    elapsed = perf_counter() - start
    
    # Dự đoán bằng công thức tuyến tính: X @ w + b
    val_preds = X_valid_scaled @ w + b
    score = evaluate(y_valid, val_preds)
    
    saved_weights[cfg["name"]] = (w, b)
    lr_rows.append({
        "config": cfg["name"], "l1": cfg["l1"], "l2": cfg["l2"],
        "best_epoch": best_ep, **score, "seconds": elapsed
    })
    print(f"[{i+1:02d}/{len(lr_configs)}] {cfg['name']:16s} | Val MAE={score['MAE']:.4f} | RMSE={score['RMSE']:.4f} | Ep={best_ep:3d} | {elapsed:.2f}s")

lr_results = pd.DataFrame(lr_rows).sort_values("MAE")
display(lr_results)

In [ ]:
# Chọn tên cấu hình L2 và L1 tốt nhất trên tập Validation
best_l2_name = lr_results[(lr_results["l2"] > 0) & (lr_results["l1"] == 0)].iloc[0]["config"]
best_l1_name = lr_results[(lr_results["l1"] > 0) & (lr_results["l2"] == 0)].iloc[0]["config"]

w_l2, b_l2 = saved_weights[best_l2_name]
w_l1, b_l1 = saved_weights[best_l1_name]

# Đọc và tạo cửa sổ trượt cho tập Test 2026
test = pd.read_csv(TEST_FILE, parse_dates=["time"]).sort_values("time").reset_index(drop=True)
combined = pd.concat([data.tail(WINDOW), test], ignore_index=True)

values_test = combined[features].to_numpy(dtype=np.float32)
temp_test = combined[TARGET].to_numpy(dtype=np.float32)

windows_test = sliding_window_view(values_test, WINDOW, axis=0)[:-1]
windows_test = windows_test.transpose(0, 2, 1)[:, ::-1, :]

X_test = windows_test.reshape(len(windows_test), -1).copy()
y_test = temp_test[WINDOW:]
persistence = temp_test[WINDOW - 1:-1]

# Chuẩn hóa X_test bằng x_mean và x_std của tập Train
X_test_scaled = ((X_test - x_mean) / x_std).astype(np.float32)

# Dự đoán trực tiếp bằng phép nhân ma trận: y_hat = X @ w + b
pred_l2 = X_test_scaled @ w_l2 + b_l2
pred_l1 = X_test_scaled @ w_l1 + b_l1

# Bảng kết quả Test
test_results = pd.DataFrame([
    {"model": "Persistence", **evaluate(y_test, persistence)},
    {"model": f"Best Ridge ({best_l2_name})", **evaluate(y_test, pred_l2)},
    {"model": f"Best Lasso ({best_l1_name})", **evaluate(y_test, pred_l1)},
]).sort_values("MAE")

display(test_results)